# L11c: Markov Decision Processes (MDPs)
A Markov Decision Process extends the basic Markov chain framework by introducing actions and rewards, enabling agents to make optimal decisions in stochastic environments. While standard Markov chains evolve passively according to fixed transition probabilities, MDPs allow an agent to choose actions that influence state transitions and generate rewards, forming the foundation for sequential decision-making under uncertainty.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Formulate a Markov decision process:__ A Markov decision process describes an agent that chooses actions, collects rewards, and moves between states with probabilities that depend on its actions. We'll define its states, actions, rewards, transition model, and discount factor, and the policy that tells the agent which action to take in each state.
> * __Solve for the optimal policy with value iteration:__ Value iteration repeatedly updates the value of each state from the values of the states it can reach next, until the values stop changing. We'll use the converged values to choose the best action in every state.
> * __Estimate values with random rollouts:__ A random rollout simulates the agent taking random actions from a starting state and adds up the discounted rewards it collects. We'll average many rollouts to estimate the value of a state without writing down the transition probabilities.

Let's get started!
___

## MDP Components
Formally, an MDP is defined by the tuple $\left(\mathcal{S}, \mathcal{A}, R\left(s, a\right), T\left(s^{\prime}\,|\,s,a\right), \gamma\right)$:

* **State space** $\mathcal{S}$: The set of all possible states $s$ the system can occupy.
* **Action space** $\mathcal{A}$: The set of all possible actions $a$ available to the agent. For a given state $s$, the available actions are denoted as $\mathcal{A}_{s} \subseteq \mathcal{A}$.
* **Reward function** $R\left(s, a\right)$: The expected immediate reward when taking action $a$ in state $s$. An individual transition produces an observed reward $r$.
* **Transition model** $T\left(s^{\prime}\,|\,s,a\right) = P(s_{t+1} = s^{\prime}\,|\,s_{t}=s,a_{t} = a)$: The probability that action $a$ in state $s$ at time $t$ results in state $s^{\prime}$ at time $t+1$.
* **Discount factor** $\gamma \in [0,1)$: A parameter that weights future rewards relative to immediate rewards, with $\gamma = 0$ prioritizing immediate rewards and $\gamma \to 1$ valuing long-term returns.

The agent's goal is to find a policy $\pi: \mathcal{S} \to \mathcal{A}$ that maps states to actions ($\pi(s) = a$) while maximizing the expected cumulative discounted reward over time, $\mathbb{E}\left[\sum_{t=0}^{\infty}\gamma^{t}R(s_{t},a_{t})\right]$.

For an episodic problem, entering a __terminal state__ ends reward collection. In value iteration, we represent a terminal state as absorbing, with a dummy action that leaves the agent there and earns zero reward. Thus, its value stays zero in every sweep initialized from zero. The reward for reaching the terminal state is collected on entry.

In this lecture, we look at two algorithms: __value iteration__, which solves for the optimal policy, and __random rollout__, which estimates the value of a state by simulation.

### Example: Grid World Navigation

Consider a robot navigating a $3\times 3$ grid world where the goal is to reach a target location while avoiding obstacles and minimizing travel time. This concrete example illustrates how the abstract MDP components work together:

* **State space** $\mathcal{S}$: The eight traversable grid positions, represented as $(x,y)$ coordinates with $x,y\in\{1,2,3\}$, excluding the obstacle at $(2,2)$. The coordinate $x$ increases to the right and $y$ increases upward.
* **Action space** $\mathcal{A}$: The set $\{\text{up}, \text{down}, \text{left}, \text{right}\}$. At boundary positions, actions that would move the agent outside the grid are unavailable, so $\mathcal{A}_s \subsetneq \mathcal{A}$ for edge and corner states.
* **Reward function** $R(s,a)$: A transition into the goal at $(3,3)$ earns $+10$, a bump into the obstacle at $(2,2)$ earns $-5$, and every other transition earns $-1$, including a slip off the grid. These are the full rewards for the respective outcomes.
* **Transition model** $T(s'|s,a)$: Actions are stochastic due to slippery terrain. The intended direction succeeds with probability $0.8$, while the two perpendicular directions each occur with probability $0.1$. For example, choosing "up" results in moving up with probability $0.8$, and moving left or right with probability $0.1$ each.
* **Discount factor** $\gamma = 0.9$: This values the $+10$ goal reward over immediate $-1$ step costs, encouraging the agent to pursue the goal despite accumulated penalties.

Because the moves are random, $R(s,a)$ is the expected reward of action $a$ in state $s$, averaged over where the agent actually moves. A move into the obstacle or off the grid leaves the agent where it is. The goal is terminal, so it earns zero further reward after entry.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-GridWorld.svg" width="900" style="max-width:100%; height:auto;" alt="Two panels. Left: a three by three grid, x increasing rightward and y upward, with the goal at (3,3) earning plus ten on entry, an obstacle at (2,2) costing minus five when bumped, and every other transition earning minus one. Right: one intended move, which succeeds with probability zero point eight and slips to each perpendicular direction with probability zero point one. A move off the grid, or a bump, leaves the robot where it is."/>
</p>

In this setting, an optimal policy $\pi^{*}$ would map each grid position to the action that maximizes expected cumulative discounted reward, accounting for both the stochastic transitions and the spatial layout of rewards and obstacles.
___

## Value Iteration
Value iteration is a dynamic programming algorithm (it builds the value of each state from the values of the states that can follow it) that computes the optimal value function $U^{*}(s)$ by iteratively applying the Bellman backup operation:

$$
\begin{equation*}
U_{k+1}(s) = \max_{a\in\mathcal{A}_s}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})\right)
\end{equation*}
$$

For $0\leq\gamma<1$, the value function converges as $k \to \infty$ such that $U_k(s) \to U^{*}(s)$. The optimal value function represents the maximum expected cumulative discounted reward achievable from each state under the best possible policy.

### Example: One Bellman Backup
Let's update the corner $s=(3,1)$. Only up and left keep the robot on the grid, and every outcome of either action earns $-1$. Starting from $U_{0}=0$, one sweep leaves the values shown below.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-GridWorld-Backup.svg" width="1100" style="max-width:100%; height:auto;" alt="Left: the grid after one sweep, every square holding its value. Right: the backup tree at the corner (3,1). Two action branches, up and left, each split into three outcomes whose edge widths show their probabilities, zero point eight for the intended move and zero point one for each slip. Every outcome earns minus one. Moving up is worth four point one one two and moving left minus one point four three two, so the new value of the corner is four point one one two and the best action is up."/>
</p>

Write $Q_{1}(s,a)$ for the expected return of taking $a$ and then using the current values $U_{1}$. Moving up reaches $(3,2)$ with probability $0.8$, slips to $(2,1)$ with probability $0.1$, and stays put with probability $0.1$. Moving left exchanges the two neighbors. With $\gamma=0.9$:

$$
\begin{aligned}
Q_1((3,1),\mathrm{up})
&=-1+0.9\bigl(0.8(7.4)+0.1(-1.4)+0.1(-1)\bigr)=4.112,\\
Q_1((3,1),\mathrm{left})
&=-1+0.9\bigl(0.8(-1.4)+0.1(7.4)+0.1(-1)\bigr)=-1.432.
\end{aligned}
$$

The backup keeps the larger, so $U_{2}(3,1)=4.112$ and the greedy action is up. Every other state updates the same way, from the previous sweep's values.

> __Where does $7.4$ come from?__ It is the same backup one square up. Moving up from $(3,2)$ reaches the goal for $+10$ with probability $0.8$, bumps the obstacle for $-5$ with probability $0.1$, and slips off the grid for $-1$ with probability $0.1$.

### Algorithm
Let's examine the value iteration pseudocode.

__Initialize__: Given an MDP with state space $\mathcal{S}$, action space $\mathcal{A}$, reward function $R(s,a)$, transition model $T\left(s^{\prime}\,|\,s,a\right)$, discount factor $\gamma$, tolerance $\epsilon>0$, and maximum number of iterations $k_{\max}$, initialize the iteration counter $k\gets 0$, the initial value function $U_{0}(s) \gets 0$ for all $s \in \mathcal{S}$, and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:
1. For each state $s \in \mathcal{S}$ __do__: compute the updated value:
   $$U_{k+1}(s) \gets \max_{a\in\mathcal{A}_s}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})\right)$$
2. Check whether to stop:
    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| \leq \epsilon$ __then__ set $\texttt{converged}\gets\texttt{true}$ and return $U^{*}\gets{U}_{k+1}$.
    - Otherwise, if $k+1\geq{k_{\max}}$ __then__ return $U_{k+1}$ with $\texttt{converged}=\texttt{false}$ and a warning that the tolerance was not met.
3. Increment the iteration counter: $k\gets{k+1}$.

__Extract Policy__: For each state $s \in \mathcal{S}$, compute:
$$\pi^{*}(s) \gets \arg\max_{a\in\mathcal{A}_s}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U^{*}}(s^{\prime})\right)$$

### Computational Complexity

Value iteration requires $O(|\mathcal{S}|^2 \cdot |\mathcal{A}|)$ operations per iteration. For each of the $|\mathcal{S}|$ states, we must evaluate $|\mathcal{A}|$ actions, and each action evaluation requires summing over $|\mathcal{S}|$ possible next states. The number of iterations until convergence depends on the discount factor $\gamma$ and the tolerance $\epsilon$, because each iteration shrinks the largest error in the value function by at least a factor of $\gamma$.

This complexity makes value iteration tractable for problems with thousands of states but challenging for continuous or very large discrete state spaces. In such cases, function approximation or sampling-based methods become necessary.

___

## Random Rollout Algorithm

Random rollout estimates $U(s)$ by simulation. Start at $s$, choose actions at random, and add up the discounted rewards. Average over many rollouts to get $\hat{U}(s)$. Nothing in that loop reads $T(s^{\prime}\,|\,s,a)$ or $R(s,a)$. The environment only has to report a next state and a reward, so rollout runs where value iteration cannot.

### Algorithm
Let's examine the random rollout pseudocode.

__Initialize__: Given an environment that can simulate actions and report next states and rewards, its state space $\mathcal{S}$ and available actions $\mathcal{A}_s$, a discount factor $0\leq\gamma<1$, a maximum depth $d\geq 1$, and a number of rollouts $N\geq 1$, initialize the value estimates $\hat{U}(s) \gets 0$ and visit counts $n(s) \gets 0$ for all $s \in \mathcal{S}$.

For $i = 1$ to $N$ __do__:
1. Set the starting state $s_{0}$ (chosen uniformly or from a start distribution), the depth counter $t\gets 0$, the cumulative return $G\gets 0$, and the current state $s\gets s_{0}$. Set $\texttt{terminated}\gets\texttt{true}$ if $s_{0}$ is terminal, and $\texttt{false}$ otherwise.
2. While $\texttt{terminated}$ is $\texttt{false}$ __do__:
    - Select action $a$ uniformly at random from $\mathcal{A}_{s}$, execute it, and observe its reward $r$ and next state $s^{\prime}$.
    - Update the cumulative return $G\gets G+\gamma^{t} r$ and the depth $t \gets t + 1$.
    - If $t \geq d$ __or__ $s^{\prime}$ is terminal __then__ set $\texttt{terminated}\gets\texttt{true}$. Otherwise, update $s\gets s^{\prime}$.
3. Increment the visit count $n(s_{0}) \gets n(s_{0}) + 1$ and update the value estimate with the incremental mean:
   $$\hat{U}(s_{0}) \gets \hat{U}(s_{0}) + \frac{1}{n(s_{0})}\left(G - \hat{U}(s_{0})\right)$$

__Output__: The estimated value $\hat{U}(s)$ for each state $s$ that served as a starting state $s_{0}$ in the $N$ rollouts.

### Convergence Properties
The average return from $s_{0}$ converges to the expected return of the __random policy__, which picks each available action with equal probability and stops after $d$ steps or at a terminal state. That is not the optimal value. Value iteration computes $U^{*}(s_{0})$; rollout computes what random behavior is worth.

A revisited state stays in the trajectory, so a bump or a return to an earlier square does not end the rollout.

The depth limit throws away whatever the trajectory would have earned after step $d$. For $|r|\leq r_{\max}$ that loss is bounded:

$$
\left|U^{\mathrm{random}}(s)-U_d^{\mathrm{random}}(s)\right|
\leq\sum_{t=d}^{\infty}\gamma^t r_{\max}
=\frac{\gamma^d r_{\max}}{1-\gamma}.
$$

Here $U^{\mathrm{random}}$ is the random policy's return without the limit and $U_d^{\mathrm{random}}$ its return with one. Depth controls this truncation error, the rollout count controls sampling error, and the two are separate knobs.

### Computational Complexity
Random rollout costs $O(N\cdot{d})$ simulated steps. Apart from one estimate per state, that cost does not grow with $|\mathcal{S}|$, which is what makes it usable on large or continuous state spaces.

Accuracy comes slowly. The standard error of a sample mean falls as $O(1/\sqrt{n(s)})$, so ten times the rollouts buys about three times the precision.

___

## Example
Let's solve the lecture's grid world with both algorithms and compare their answers.

> [▶ Value iteration and random rollout in the slippery grid world](CHEME-5800-L11c-Example-RandomRollout-Fall-2026.ipynb). How close do random rollouts come to the values that value iteration computes? We build the slippery $3\times3$ grid world, check it against the one Bellman backup we worked by hand, and solve it with value iteration. We then estimate the value of each square by averaging random rollouts and compare the estimates with the exact value of the random policy and with the optimal values. The comparison shows that a rollout measures how well random actions do, and that its standard error shrinks slowly as the rollouts accumulate.

___

## Lab
In the [lab](../L11d/CHEME-5800-L11d-Lab-ValueIterationGridWorld-Fall-2026.ipynb), we build a grid world in which a robot must reach a charging station while avoiding lava pits, write our own value iteration with a convergence check, and use the optimal policy to plan the robot's path.

___

## Summary
In this lecture, we explored how Markov decision processes extend Markov chains by adding actions and rewards to enable optimal sequential decision-making:

> __Key Takeaways:__
>
> * __MDPs add actions and rewards to Markov chains:__ We added actions that change the transition probabilities and rewards that score each choice, so the agent's decisions now shape where the process goes. The discount factor sets how much the agent values future rewards against immediate ones, and a policy assigns an action to every state.
>
> * __Value iteration finds the optimal policy:__ We applied the Bellman backup to every state until the values stopped changing, which gives the maximum expected discounted reward from each state when the discount factor is less than one. Choosing the action that achieves that maximum in each state gives the optimal policy.
>
> * __Random rollout estimates values by simulation:__ We simulated random trajectories until termination or a depth limit and averaged their observed discounted rewards. The depth limit controls the omitted future return, while the number of rollouts controls sampling uncertainty. This needs only a way to simulate the environment, not its transition probabilities, and it tells us how well random actions do, not the optimal value.

These foundational concepts enable practical applications in robotics, game playing, resource allocation, and autonomous systems where agents must make sequential decisions under uncertainty.

___